# CountZES — Run Pipeline on Google Colab

Notebook dùng để cài môi trường, tải weights/dataset và chạy pipeline đánh giá chuẩn của **CountZES** trên GPU Colab.

**Trước khi chạy:**
1. Trên Colab Web: chọn **Runtime → Change runtime type → T4 GPU**.
2. Trong Antigravity IDE (với extension `Google.colab`): bấm **Select Kernel → Colab → Auto Connect**.
3. Chạy lần lượt từng cell từ trên xuống dưới.

In [ ]:
# [CELL-1] Kiểm tra GPU và thiết lập biến môi trường PyTorch
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

!nvidia-smi
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# [CELL-2] Clone hoặc cập nhật code mới nhất từ GitHub của bạn
REPO_URL = "https://github.com/duylw/cv-advanced.git"
REPO_DIR = "/content/cv-advanced"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin main
!git log -n 2 --oneline

In [ ]:
# [CELL-3] Cài các thư viện cần thiết (tương thích Python/PyTorch sẵn có của Colab)
!pip install -q opencv-python tqdm matplotlib timm einops supervision sentencepiece accelerate scipy pandas scikit-image addict yapf pycocotools gdown
!pip install -q "transformers>=4.31.0"
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Kiểm tra nhanh cú pháp toàn bộ code pipeline
!python -m py_compile countzes_core.py countzes_fsc147.py countzes_carpk.py countzes_persense_D.py countzes_mbm.py countzes_vgg.py

In [ ]:
# [CELL-4] (Tùy chọn) Mount Google Drive để lưu weights dùng lại giữa các phiên Colab
USE_GDRIVE_CACHE = False  # Đổi thành True nếu muốn lưu weights trên Google Drive
DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/CountZES_weights"

if USE_GDRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)
    print("Google Drive cache dir:", DRIVE_WEIGHTS_DIR)

In [ ]:
# [CELL-5] Tải Model Weights (SAM ViT-H, DSALVANet, GroundingDINO)
import shutil

os.makedirs("data", exist_ok=True)
os.makedirs("DSALVANet/checkpoints", exist_ok=True)
os.makedirs("GroundingDINO/weights", exist_ok=True)
os.makedirs("CounTR/output_allnew_dir", exist_ok=True)

def fetch_with_cache(local_path, download_cmd):
    cache_path = os.path.join(DRIVE_WEIGHTS_DIR, os.path.basename(local_path)) if USE_GDRIVE_CACHE else None
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        print(f"[OK] Đã có sẵn: {local_path}")
        return
    if cache_path and os.path.exists(cache_path) and os.path.getsize(cache_path) > 0:
        os.symlink(cache_path, local_path)
        print(f"[Drive Symlink] {local_path} -> {cache_path}")
        return
    get_ipython().system(download_cmd)
    if cache_path and os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        shutil.copy2(local_path, cache_path)
        print(f"[Saved to Drive] {cache_path}")

# 1. SAM ViT-H (~2.56 GB)
fetch_with_cache(
    "data/sam_vit_h_4b8939.pth",
    "wget -q --show-progress -O data/sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
)

# 2. DSALVANet pretrained FSC-147
fetch_with_cache(
    "DSALVANet/checkpoints/checkpoint_200.pth",
    "gdown 1julzH9MJSK1xTGchb1r0CXdZ2wzF5-Kp -O DSALVANet/checkpoints/checkpoint_200.pth"
)

# 3. GroundingDINO SwinT-OGC
fetch_with_cache(
    "GroundingDINO/weights/groundingdino_swint_ogc.pth",
    "gdown 13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g -O GroundingDINO/weights/groundingdino_swint_ogc.pth"
)

print("\nNếu chạy với --fsoc countr, hãy đặt thêm FSC147.pth / CARPK.pth vào CounTR/output_allnew_dir/")

In [ ]:
# [CELL-6] Tải dataset mẫu (VGG / MBM) để chạy thử pipeline
DATASET_TO_FETCH = "vgg"  # "vgg" hoặc "mbm"

if DATASET_TO_FETCH == "vgg" and not os.path.exists("data/vgg/Images"):
    !gdown 1FerZjhpA1MCtsmjD-eF1byZvOWEDnwZL -O /content/vgg.zip
    !mkdir -p data/vgg && unzip -q -o /content/vgg.zip -d data/vgg
elif DATASET_TO_FETCH == "mbm" and not os.path.exists("data/mbm/Images"):
    !gdown 1i54Hcw2GbCVp0Kz7jei2Lr7Cmi2DZkol -O /content/mbm.zip
    !mkdir -p data/mbm && unzip -q -o /content/mbm.zip -d data/mbm

# Kiểm tra cấu trúc thư mục sau khi giải nén (cần có dạng data/<dataset>/Images/<class>/<id>.png)
!find data -maxdepth 4

In [ ]:
# [CELL-7] Chạy pipeline đánh giá chính thức
SCRIPT = "countzes_vgg.py"       # countzes_vgg.py | countzes_mbm.py | countzes_fsc147.py | countzes_carpk.py | countzes_persense_D.py
FSOC = "DSALVANet"               # "DSALVANet" hoặc "countr" (countr cần FSC147.pth / CARPK.pth)
OUTDIR = "colab_run_default"
EXTRA_FLAGS = "--visualize"      # Thêm "--fix_sses_inversion --single_instance_mode unit" để A/B test P2

!python {SCRIPT} --fsoc {FSOC} --outdir {OUTDIR} {EXTRA_FLAGS}

In [ ]:
# [CELL-8] Xem nhanh ảnh kết quả từ outputs/
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob(f"outputs/{OUTDIR}/**/*.png", recursive=True))[:6]
print(f"Tìm thấy {len(imgs)} ảnh hiển thị mẫu:")
for p in imgs:
    print(p)
    display(Image(filename=p))